# Module 00: System Design Fundamentals Interview Playbook — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/interview_capacity_calculator.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import interview_capacity_calculator

classes = [n for n, o in inspect.getmembers(interview_capacity_calculator, inspect.isclass)
           if o.__module__ == 'interview_capacity_calculator']
functions = [n for n, o in inspect.getmembers(interview_capacity_calculator, inspect.isfunction)
             if o.__module__ == 'interview_capacity_calculator']

print('module   : interview_capacity_calculator')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(interview_capacity_calculator, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Qps calculation

This is the module's own `test_qps_calculation` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
from interview_capacity_calculator import CapacityCalculator, SystemCapacityProfile


def twitter_profile() -> SystemCapacityProfile:
    """Sample profile representing Twitter-like scale:
    500M DAU, 20 reads/day, 2 writes/day, 200B text payload.
    """
    return SystemCapacityProfile(
        dau=500_000_000,
        reads_per_user_day=20.0,
        writes_per_user_day=2.0,
        read_payload_bytes=2_000,  # 2 KB timeline response
        write_payload_bytes=250,   # 250 B tweet metadata
        peak_multiplier=2.5,
        storage_years=5,
        cache_read_ratio=0.20,
    )

_make_twitter_profile = twitter_profile

twitter_profile = _make_twitter_profile()

qps = CapacityCalculator.calculate_qps(twitter_profile)
# Writes: 500M * 2 = 1B / 86400 = ~11,574.07 QPS
assert qps["avg_write_qps"] == 11574.07
assert qps["peak_write_qps"] == 28935.19
# Reads: 500M * 20 = 10B / 86400 = ~115,740.74 QPS
assert qps["avg_read_qps"] == 115740.74
assert qps["peak_read_qps"] == 289351.85

print('PASSED: test_qps_calculation')

## 3. 🔮 Prediction — commit before you run

You are asked to size a system for 10 million daily active users. Before running the next cell, write down your estimate for peak QPS. What multiplier over the daily average did you apply, and why is a flat 1x wrong?

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_bandwidth_calculation`, which tests exactly this property.


In [ ]:
def twitter_profile() -> SystemCapacityProfile:
    """Sample profile representing Twitter-like scale:
    500M DAU, 20 reads/day, 2 writes/day, 200B text payload.
    """
    return SystemCapacityProfile(
        dau=500_000_000,
        reads_per_user_day=20.0,
        writes_per_user_day=2.0,
        read_payload_bytes=2_000,  # 2 KB timeline response
        write_payload_bytes=250,   # 250 B tweet metadata
        peak_multiplier=2.5,
        storage_years=5,
        cache_read_ratio=0.20,
    )

_make_twitter_profile = twitter_profile

twitter_profile = _make_twitter_profile()

bw = CapacityCalculator.calculate_bandwidth(twitter_profile)
# Ingress: 11,574.07 * 250 B = ~2.89 MB/s = ~23.15 Mbps
assert bw["ingress_mb_per_sec"] == 2.89
assert bw["ingress_mbps"] == 23.15
# Egress: 115,740.74 * 2000 B = ~231.48 MB/s = ~1851.85 Mbps
assert bw["egress_mb_per_sec"] == 231.48
assert bw["egress_mbps"] == 1851.85

print('PASSED: test_bandwidth_calculation')

## 4. Measure it: Storage calculation

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_storage_calculation` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

def twitter_profile() -> SystemCapacityProfile:
    """Sample profile representing Twitter-like scale:
    500M DAU, 20 reads/day, 2 writes/day, 200B text payload.
    """
    return SystemCapacityProfile(
        dau=500_000_000,
        reads_per_user_day=20.0,
        writes_per_user_day=2.0,
        read_payload_bytes=2_000,  # 2 KB timeline response
        write_payload_bytes=250,   # 250 B tweet metadata
        peak_multiplier=2.5,
        storage_years=5,
        cache_read_ratio=0.20,
    )

_make_twitter_profile = twitter_profile

twitter_profile = _make_twitter_profile()

storage = CapacityCalculator.calculate_storage(twitter_profile)
# Daily writes: 1B * 250 B = 250 GB/day
assert storage["daily_storage_gb"] == 250.0
# Annual: 250 GB * 365 = 91.25 TB/year
assert storage["annual_storage_tb"] == 91.25
# 5-Year: 91.25 * 5 = 456.25 TB
assert storage["multi_year_storage_tb"] == 456.25

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_storage_calculation')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(interview_capacity_calculator) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Capacity estimates are decisions, not trivia - state your assumptions out loud.
2. Peak is a multiple of average; the multiplier is the interesting number.
3. An interview answer with no numbers in it is not an answer.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
